# Reddit Kafka Sentiment Analysis Consumer

This notebook consumes Reddit posts from Kafka, performs sentiment analysis on both post content and comments using CryptoBERT, and holds the enriched data.

## Architecture
```
Reddit Posts + Comments → Kafka (reddit-raw) → Sentiment Analysis (CryptoBERT) → Enhanced Data Storage
```

### Configuration:
- **Kafka Broker**: `0.tcp.ap.ngrok.io:14089`
- **Topic**: `reddit-raw`
- **Model**: CryptoBERT for crypto sentiment analysis
- **Scope**: Analyzes post content + all associated comments

## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install pyspark findspark
!pip install torch transformers accelerate
!pip install pandas numpy

print("✅ All packages installed successfully")

In [ ]:
import os
import re
import json
import time
import logging
import hashlib
from datetime import datetime, timezone
from typing import Dict, Any, List

# Data processing
import pandas as pd
import numpy as np
from collections import Counter

# Spark
import findspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, lit, pandas_udf
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, FloatType, BooleanType

# ML and Sentiment Analysis
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✅ All imports successful")

## 2. Configuration

In [ ]:
# Kafka Configuration
KAFKA_BROKER = "0.tcp.ap.ngrok.io:14089"
TOPIC_NAME = "reddit-raw"

# Model Configuration (following CryptoBERT documentation)
MODEL_NAME = 'ElKulako/cryptobert'
MAX_SEQUENCE_LENGTH = 64  # Recommended length from model docs

# Sentiment label mapping (CryptoBERT)
LABEL_MAP = {0: 'Bearish', 1: 'Neutral', 2: 'Bullish'}  # Based on typical BERT sentiment models

# Data storage for processed Reddit posts with sentiment
processed_reddit_data = []

print(f"🔧 Configuration:")
print(f"   - Kafka Broker: {KAFKA_BROKER}")
print(f"   - Topic: {TOPIC_NAME}")
print(f"   - Model: {MODEL_NAME}")
print(f"   - Max Sequence Length: {MAX_SEQUENCE_LENGTH} (per model docs)")
print(f"   - Data storage: In-memory list for enriched data")

## 3. Text Preprocessing Function

In [ ]:
def preprocess_text(text: str) -> str:
    """Preprocess text for sentiment analysis (adapted from CryptoBERT approach)"""
    if not text or not isinstance(text, str):
        return ""
    
    # Convert to lowercase
    text = text.lower().strip()
    
    # Remove URLs
    text = re.sub(r'http[s]?://\S+', '', text)
    
    # Remove Reddit-specific patterns
    text = re.sub(r'r/\w+', '', text)  # Remove subreddit references
    text = re.sub(r'u/\w+', '', text)  # Remove user references
    
    # Remove mentions and hashtags
    text = re.sub(r'[@#]\w+', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Remove non-alphanumeric chars except basic punctuation
    text = re.sub(r'[^\w\s.,!?;:]', '', text)
    
    return text

## 4. Load CryptoBERT Model (BertSentiment Approach)

In [ ]:
# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Using device: {device}")

# Load CryptoBERT model and tokenizer
print(f"📥 Loading model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)

print("✅ Model loaded successfully!")

## 5. Sentiment Prediction Function

In [ ]:
def predict_sentiment_batch(texts: List[str]) -> List[Dict[str, Any]]:
    """Predict sentiment for a batch of texts using CryptoBERT"""
    results = []
    
    for text in texts:
        # Preprocess text
        clean_text = preprocess_text(text)
        
        # Tokenize with recommended max_length = 64
        inputs = tokenizer(
            clean_text,
            truncation=True,
            padding=True,
            max_length=MAX_SEQUENCE_LENGTH,
            return_tensors='pt'
        ).to(device)
        
        # Predict
        with torch.no_grad():
            outputs = model(**inputs)
            probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
            predicted_class = torch.argmax(probabilities, dim=-1).item()
            confidence_score = torch.max(probabilities).item()
        
        # Convert to sentiment label
        sentiment_label = LABEL_MAP.get(predicted_class, 'Neutral')
        
        results.append({
            'sentiment': sentiment_label,
            'score': float(confidence_score),
            'processed_text': clean_text
        })
    
    return results

def analyze_reddit_post_with_comments(post_data: Dict) -> Dict:
    """Analyze sentiment for both post content and all comments"""
    # Analyze post sentiment
    post_sentiment = predict_sentiment_batch([post_data.get('text_clean', '')])[0]
    
    # Create enhanced post data
    enhanced_post = post_data.copy()
    enhanced_post['sentiment_analysis'] = post_sentiment
    
    # Analyze comments if they exist
    if 'comments' in post_data and post_data['comments']:
        enhanced_comments = []
        
        for comment in post_data['comments']:
            # Analyze comment sentiment
            comment_sentiment = predict_sentiment_batch([comment.get('text_clean', '')])[0]
            
            # Create enhanced comment
            enhanced_comment = comment.copy()
            enhanced_comment['sentiment_analysis'] = comment_sentiment
            enhanced_comments.append(enhanced_comment)
        
        enhanced_post['comments'] = enhanced_comments
        
        # Calculate overall sentiment statistics for the post
        if enhanced_comments:
            # Aggregate comment sentiments
            comment_sentiments = [c['sentiment_analysis']['sentiment'] for c in enhanced_comments]
            sentiment_counts = Counter(comment_sentiments)
            
            # Calculate average score
            avg_score = sum(c['sentiment_analysis']['score'] for c in enhanced_comments) / len(enhanced_comments)
            
            enhanced_post['comment_sentiment_stats'] = {
                'total_comments': len(enhanced_comments),
                'sentiment_distribution': dict(sentiment_counts),
                'average_score': float(avg_score),
                'bullish_percentage': float((sentiment_counts.get('Bullish', 0) / len(enhanced_comments)) * 100),
                'bearish_percentage': float((sentiment_counts.get('Bearish', 0) / len(enhanced_comments)) * 100),
                'neutral_percentage': float((sentiment_counts.get('Neutral', 0) / len(enhanced_comments)) * 100)
            }
    
    return enhanced_post

## 6. Setup Spark Streaming

In [ ]:
# Stop any existing Spark session
try:
    spark.stop()
except:
    pass

# Spark configuration
scala_version = '2.12'
spark_version = '3.5.1'
packages = [
    f'org.apache.spark:spark-sql-kafka-0-10_{scala_version}:{spark_version}',
    'org.apache.kafka:kafka-clients:3.5.1'
]

# Initialize Spark
findspark.init()
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("RedditKafkaSentimentConsumer") \
    .config("spark.jars.packages", ",".join(packages)) \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("✅ Spark session created successfully")

## 7. Define Reddit Data Schema

In [ ]:
# Define schema for full Reddit JSON data (posts + comments)
# We'll handle the full JSON structure as string and parse it in Python
reddit_schema = StructType([
    StructField("json_data", StringType(), True)  # Full Reddit JSON as string
])

print("📋 Reddit data schema defined:")
print("   - json_data: Full Reddit JSON structure (metadata + posts array)")
print("   - Will parse posts and comments in Python for sentiment analysis")

## 8. Create Kafka Stream

In [ ]:
# Create Kafka streaming DataFrame
df_raw = (spark
    .readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BROKER)
    .option("subscribe", TOPIC_NAME)
    .option("startingOffsets", "latest")
    .load())

print(f"📡 Created Kafka stream from topic '{TOPIC_NAME}' at {KAFKA_BROKER}")
df_raw.printSchema()

## 9. Parse Reddit JSON Data

In [ ]:
# Parse JSON data from Kafka - get full Reddit structure
df_reddit = (df_raw
    .selectExpr("CAST(value AS STRING) as json_data")
    .filter(col("json_data").isNotNull())
)

print("🔄 Reddit data parsing pipeline created")
print("   - Will process full Reddit JSON structure with posts and comments")
df_reddit.printSchema()

## 10. Create Sentiment Analysis UDF

In [ ]:
def process_reddit_json_batch(json_data_list):
    """Process a batch of Reddit JSON data and add sentiment analysis"""
    enhanced_posts = []
    
    for json_str in json_data_list:
        # Parse Reddit JSON
        reddit_data = json.loads(json_str)
        
        # Process each post in the data
        if 'posts' in reddit_data:
            for post in reddit_data['posts']:
                # Analyze post and all its comments
                enhanced_post = analyze_reddit_post_with_comments(post)
                enhanced_posts.append(enhanced_post)
    
    return enhanced_posts

## 11. Apply Sentiment Analysis

In [ ]:
print("✅ Ready for real-time Reddit sentiment analysis processing")

## 12. Define Batch Processing Function

In [ ]:
def foreach_batch_function(df, batch_id):
    """Process each batch of Reddit JSON data with sentiment analysis"""
    global processed_reddit_data
    
    # Collect JSON data from the batch
    json_data = [row.json_data for row in df.collect()]
    
    print(f"\n{'='*60}")
    print(f"🔄 Processing Batch {batch_id} ({len(json_data)} JSON messages)")
    
    # Process Reddit data with sentiment analysis
    enhanced_posts = process_reddit_json_batch(json_data)
    
    # Store processed data
    processed_reddit_data.extend(enhanced_posts)
    
    print(f"✅ Enhanced {len(enhanced_posts)} posts with sentiment analysis")
    print(f"📊 Total processed posts so far: {len(processed_reddit_data)}")
    
    # Overall sentiment distribution for posts
    post_sentiments = [post['sentiment_analysis']['sentiment'] for post in enhanced_posts]
    post_sentiment_counts = Counter(post_sentiments)
    
    print(f"\n📈 Batch {batch_id} Analysis Summary:")
    print(f"   📝 Post Sentiment Distribution:")
    for sentiment, count in post_sentiment_counts.items():
        percentage = (count / len(enhanced_posts)) * 100
        print(f"      • {sentiment}: {count} ({percentage:.1f}%)")
    
    # Average score for posts
    avg_post_score = sum(post['sentiment_analysis']['score'] for post in enhanced_posts) / len(enhanced_posts)
    print(f"   🎯 Average Post Score: {avg_post_score:.3f}")
    
    # Comments analysis
    total_comments = sum(len(post.get('comments', [])) for post in enhanced_posts)
    if total_comments > 0:
        print(f"   💬 Total Comments Analyzed: {total_comments}")
        
        all_comment_sentiments = []
        all_comment_scores = []
        
        for post in enhanced_posts:
            for comment in post.get('comments', []):
                if 'sentiment_analysis' in comment:
                    all_comment_sentiments.append(comment['sentiment_analysis']['sentiment'])
                    all_comment_scores.append(comment['sentiment_analysis']['score'])
        
        if all_comment_sentiments:
            comment_sentiment_counts = Counter(all_comment_sentiments)
            avg_comment_score = sum(all_comment_scores) / len(all_comment_scores)
            
            print(f"   💬 Comment Sentiment Distribution:")
            for sentiment, count in comment_sentiment_counts.items():
                percentage = (count / len(all_comment_sentiments)) * 100
                print(f"      • {sentiment}: {count} ({percentage:.1f}%)")
            
            print(f"   🎯 Average Comment Score: {avg_comment_score:.3f}")
    
    # Show sample posts
    print(f"\n📋 Enhanced Posts in this batch:")
    for i, post in enumerate(enhanced_posts[:3], 1):
        text_preview = post['text_clean'][:80] + "..." if len(post['text_clean']) > 80 else post['text_clean']
        print(f"   {i}. [{post['sentiment_analysis']['sentiment']}] {text_preview}")
        print(f"      Score: {post['sentiment_analysis']['score']:.3f}")
        print(f"      Comments: {len(post.get('comments', []))}")
    
    print(f"{'='*60}")

print("✅ Real-time batch processing function ready")

## 13. Start Streaming Query

In [ ]:
print("🚀 Starting Reddit sentiment analysis stream...")
print(f"📡 Listening to Kafka topic: {TOPIC_NAME}")
print(f"🔗 Kafka broker: {KAFKA_BROKER}")
print(f"🤖 Model: {MODEL_NAME}")
print(f"📊 Processing: Posts + Comments with sentiment analysis")
print(f"💾 Storage: In-memory list of enhanced Reddit data")
print("\n⏳ Waiting for Reddit posts... (Press Ctrl+C to stop)\n")

# Start streaming query with the full JSON structure
query = (df_reddit
    .writeStream
    .foreachBatch(foreach_batch_function)
    .trigger(processingTime="5 seconds")
    .option("checkpointLocation", "/tmp/reddit_full_sentiment_checkpoint")
    .start()
)

print(f"✅ Stream started successfully!")
print(f"📊 Query ID: {query.id}")
print(f"🔍 Query status: {query.status['message']}")

## 14. Run Stream with Timeout

In [ ]:
# Run the stream for a specified time or until interrupted
try:
    # Wait for the stream to process data (5 minutes max)
    query.awaitTermination(timeout=300)  
except KeyboardInterrupt:
    print("\n⏹️  Stream interrupted by user.")
except Exception as e:
    print(f"\n❌ Stream error: {e}")
finally:
    # Ensure proper cleanup
    if 'query' in locals() and query.isActive:
        print("\n🛑 Stopping query...")
        query.stop()
        print("✅ Query stopped successfully.")
    
    # Show final statistics if available
    if 'query' in locals() and hasattr(query, 'lastProgress') and query.lastProgress:
        print(f"\n📈 Final Statistics:")
        progress = query.lastProgress
        print(f"   - Total input rows: {progress.get('numInputRows', 0)}")
        print(f"   - Input rows per second: {progress.get('inputRowsPerSecond', 0):.2f}")
        print(f"   - Processing time per batch: {progress.get('processingTimeMsPerBatch', 0)}ms")

## 15. Cleanup and Summary

In [ ]:
# Display all stored Reddit data with sentiment analysis
def display_stored_reddit_data():
    """Display all processed Reddit data stored in memory"""
    print(f"\n🗂️  STORED REDDIT DATA WITH SENTIMENT ANALYSIS")
    print(f"{'='*80}")
    
    if not processed_reddit_data:
        print("❌ No Reddit data has been processed yet.")
        return
    
    print(f"📊 Total Posts Processed: {len(processed_reddit_data)}")
    
    # Overall statistics
    all_post_sentiments = [post['sentiment_analysis']['sentiment'] for post in processed_reddit_data]
    all_post_scores = [post['sentiment_analysis']['score'] for post in processed_reddit_data]
    
    print(f"\n📈 OVERALL ANALYSIS:")
    print(f"   • Average Post Score: {sum(all_post_scores)/len(all_post_scores):.3f}")
    print(f"   • High Score Posts (>0.8): {sum(1 for s in all_post_scores if s > 0.8)}/{len(all_post_scores)} ({(sum(1 for s in all_post_scores if s > 0.8)/len(all_post_scores)*100):.1f}%)")
    
    # Post sentiment distribution
    post_sentiment_counts = Counter(all_post_sentiments)
    print(f"\n📝 POST SENTIMENT DISTRIBUTION:")
    for sentiment, count in post_sentiment_counts.items():
        percentage = (count / len(all_post_sentiments)) * 100
        print(f"   • {sentiment}: {count} ({percentage:.1f}%)")
    
    # Comments analysis
    total_comments = sum(len(post.get('comments', [])) for post in processed_reddit_data)
    if total_comments > 0:
        all_comment_sentiments = []
        all_comment_scores = []
        
        for post in processed_reddit_data:
            for comment in post.get('comments', []):
                if 'sentiment_analysis' in comment:
                    all_comment_sentiments.append(comment['sentiment_analysis']['sentiment'])
                    all_comment_scores.append(comment['sentiment_analysis']['score'])
        
        if all_comment_sentiments:
            print(f"\n💬 COMMENT ANALYSIS:")
            print(f"   • Total Comments: {len(all_comment_sentiments)}")
            print(f"   • Average Comment Score: {sum(all_comment_scores)/len(all_comment_scores):.3f}")
            
            comment_sentiment_counts = Counter(all_comment_sentiments)
            print(f"\n💬 COMMENT SENTIMENT DISTRIBUTION:")
            for sentiment, count in comment_sentiment_counts.items():
                percentage = (count / len(all_comment_sentiments)) * 100
                print(f"   • {sentiment}: {count} ({percentage:.1f}%)")
    
    # Show detailed posts
    print(f"\n📋 DETAILED POSTS WITH SENTIMENT:")
    print(f"{'='*80}")
    
    for i, post in enumerate(processed_reddit_data, 1):
        print(f"\n📌 POST {i}: {post['id']}")
        print(f"   Platform: {post.get('platform', 'Unknown')}")
        print(f"   Author: {post.get('author_id', 'Unknown')}")
        print(f"   Created: {post.get('post_created_timestamp', 'Unknown')}")
        print(f"   Language: {post.get('lang', 'Unknown')}")
        
        # Post content and sentiment
        print(f"\n   📝 POST CONTENT:")
        print(f"   Text: {post['text_clean']}")
        print(f"   Sentiment: {post['sentiment_analysis']['sentiment']}")
        print(f"   Score: {post['sentiment_analysis']['score']:.3f}")
        
        # Comments with sentiment
        if 'comments' in post and post['comments']:
            print(f"\n   💬 COMMENTS ({len(post['comments'])}):")
            for j, comment in enumerate(post['comments'], 1):
                print(f"   {j}. [{comment['sentiment_analysis']['sentiment']}] {comment['text_clean']}")
                print(f"      Score: {comment['sentiment_analysis']['score']:.3f}")
                print(f"      Author: {comment.get('author_id', 'Unknown')}")
        
        # Comment sentiment statistics
        if 'comment_sentiment_stats' in post:
            stats = post['comment_sentiment_stats']
            print(f"\n   📊 COMMENT STATISTICS:")
            print(f"   • Total: {stats['total_comments']}")
            print(f"   • Sentiment Distribution: {stats['sentiment_distribution']}")
            print(f"   • Average Score: {stats['average_score']:.3f}")
            print(f"   • Bullish %: {stats['bullish_percentage']:.1f}%")
            print(f"   • Bearish %: {stats['bearish_percentage']:.1f}%")
            print(f"   • Neutral %: {stats['neutral_percentage']:.1f}%")
        
        print(f"   {'-'*60}")

# Call the display function
display_stored_reddit_data()